In [11]:
"""
Shared paths for this notebook.

Reads recordings from Box Drive. Writes derived files locally so we do not
sync huge .npz outputs back to Box.

Run cells in order. Stage 2/3 read full LFPs — keep PROCESS_LIMIT small until
a smoke test succeeds, then set it to None to process every file.
"""

from pathlib import Path

DATA_ROOT = Path("/Users/prestondavis/Library/CloudStorage/Box-Box/LFP Only Data")
PROJECT = Path("/Users/prestondavis/Desktop/Capstone")
DERIVED = PROJECT / "derived"
FIGURES = PROJECT / "figures"
FIGURES_CLEAN = PROJECT / "figures_clean"
RESULTS = PROJECT / "results"

# First N files for preprocess + features. None = all 203.
PROCESS_LIMIT = None
STAT_METRIC = "burst_rate_per_min"
EXCLUDE = True

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Box data folder not found:\n  {DATA_ROOT}\n"
        "Is Box Drive running and signed in?"
    )

for p in (DERIVED, DERIVED / "clean", FIGURES, FIGURES_CLEAN, RESULTS):
    p.mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / "python"))

print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"DERIVED        = {DERIVED}")
print(f"PROCESS_LIMIT  = {PROCESS_LIMIT}")
print(f"EXCLUDE        = {EXCLUDE}")


DATA_ROOT      = /Users/prestondavis/Library/CloudStorage/Box-Box/LFP Only Data
DERIVED        = /Users/prestondavis/Desktop/Capstone/derived
PROCESS_LIMIT  = None


In [12]:
"""
Stage 1 -- Build and validate the dataset manifest.

Walks the LFP Only Data tree (or the exported HDF5 tree), parses every filename,
constructs the correct subject key per cohort, and runs assertions against the
expected inventory. Nothing downstream should ever re-parse a filename.

Run:
    python step1_manifest.py --root "/path/to/LFP Only Data" --out ../derived

Key idea you must not lose:
    C1/C2 are WITHIN-subject crossovers -- the same mouse appears under psi and
    sal. Its subject key must NOT include the drug.
    C3/C4 are BETWEEN-subject -- each mouse belongs to exactly one group.
"""

from __future__ import annotations

import argparse
import sys
from pathlib import Path

import pandas as pd

RECTYPES = {"BL1", "inj", "EZM", "EPM", "BL2"}

# Expected counts, from the README and verified against the Box folder.
EXPECTED_FILES_PER_COHORT = {"C1": 56, "C2": 44, "C3": 60, "C4": 43}
EXPECTED_TOTAL = 203

KNOWN_GAPS = [
    ("C2", "F2", "saline sessions only; no psilocybin week"),
    ("C4", "sal_sal_M1", "no EPM recording"),
]


def parse_stem(stem: str) -> dict:
    """Psi_C<N>_<group tokens...>_<mouse>_<rectype> -> dict."""
    p = stem.split("_")
    if len(p) < 5 or p[0] != "Psi":
        raise ValueError(f"unexpected filename pattern: {stem}")
    return {
        "stem": stem,
        "cohort": p[1],
        "group": "_".join(p[2:-2]),
        "mouse": p[-2],
        "rectype": p[-1],
    }


def enrich(row: dict) -> dict:
    """Add derived design columns. This is where the design logic lives."""
    cohort, group, mouse = row["cohort"], row["group"], row["mouse"]

    within = cohort in ("C1", "C2")
    row["design"] = "within" if within else "between"

    # Subject key: the thing that identifies an ANIMAL.
    row["subject_uid"] = f"{cohort}_{mouse}" if within else f"{cohort}_{group}_{mouse}"

    # Session key: one behavioral test day (a mouse-week for the crossovers).
    row["session_uid"] = f"{cohort}_{group}_{mouse}"

    # Decompose group into the two pharmacological factors, uniformly across
    # cohorts, so C3 and C4 can be pooled into one 2x2 frame later.
    toks = group.split("_")
    if len(toks) == 2:                      # C4: blocker_drug
        blocker, drug = toks
    elif cohort in ("C1", "C2"):            # no blocker used at all
        blocker, drug = "none", group
    else:                                   # C3 single-token arms
        blocker, drug = {
            "sal": ("sal", "sal"),
            "psi": ("sal", "psi"),
            "ket": ("ket", "psi"),
        }.get(group, ("unknown", "unknown"))

    row["blocker"] = blocker
    row["drug"] = drug
    row["sex"] = mouse[0] if mouse[:1] in ("F", "M") else "?"
    row["animal_number"] = mouse[1:]
    row["behavior_assay"] = "EZM" if cohort in ("C1", "C2") else "EPM"
    row["is_baseline"] = row["rectype"] in ("BL1", "BL2")
    row["is_behavior"] = row["rectype"] in ("EZM", "EPM")
    return row


def build(root: Path) -> pd.DataFrame:
    paths = sorted(root.rglob("*.mat")) or sorted(root.rglob("*.h5"))
    if not paths:
        sys.exit(f"No .mat or .h5 files found under {root}")

    rows = []
    for p in paths:
        row = parse_stem(p.stem)
        row["path"] = str(p)
        row["size_mb"] = round(p.stat().st_size / 1e6, 2)
        rows.append(enrich(row))

    df = pd.DataFrame(rows)
    cols = [
        "stem", "cohort", "group", "blocker", "drug", "mouse", "sex",
        "rectype", "design", "behavior_assay", "subject_uid", "session_uid",
        "is_baseline", "is_behavior", "size_mb", "path",
    ]
    return df[cols].sort_values(["cohort", "group", "mouse", "rectype"]).reset_index(drop=True)


def validate(df: pd.DataFrame) -> list[str]:
    """Assertions that should hold. Returns a list of problems (empty = good)."""
    problems = []

    if len(df) != EXPECTED_TOTAL:
        problems.append(f"expected {EXPECTED_TOTAL} files, found {len(df)}")

    for cohort, n in EXPECTED_FILES_PER_COHORT.items():
        got = int((df.cohort == cohort).sum())
        if got != n:
            problems.append(f"{cohort}: expected {n} files, found {got}")

    bad = set(df.rectype) - RECTYPES
    if bad:
        problems.append(f"unexpected recording types: {sorted(bad)}")

    if (df.sex == "?").any():
        problems.append(f"unparseable mouse IDs: {sorted(set(df.loc[df.sex=='?','mouse']))}")

    if (df.blocker == "unknown").any():
        problems.append(f"ungrouped drug tokens: {sorted(set(df.loc[df.blocker=='unknown','group']))}")

    # Every session should have all 4 recording types -- except the known gap.
    per_session = df.groupby("session_uid").rectype.nunique()
    incomplete = per_session[per_session < 4].index.tolist()
    unexpected = [s for s in incomplete if s != "C4_sal_sal_M1"]
    if unexpected:
        problems.append(f"incomplete sessions beyond the known gap: {unexpected}")

    # Cohort-1/2 crossover completeness: how many mice have both drugs?
    for cohort in ("C1", "C2"):
        sub = df[df.cohort == cohort]
        per_mouse = sub.groupby("mouse").drug.nunique()
        singles = per_mouse[per_mouse < 2].index.tolist()
        if singles:
            problems.append(f"{cohort}: mice with only one drug week: {singles} "
                            f"(F2 in C2 is expected and documented)")
    return problems


def summarize(df: pd.DataFrame) -> None:
    print("\n=== Files per cohort x recording type ===")
    print(pd.crosstab(df.cohort, df.rectype))

    print("\n=== ANIMALS per cohort x group (this is your real n) ===")
    n_animals = (df.groupby(["cohort", "group"]).subject_uid.nunique()
                   .rename("n_mice").reset_index())
    print(n_animals.to_string(index=False))

    print("\n=== Pooled 2x2 frame for C3+C4 (animals) ===")
    epm = df[df.cohort.isin(["C3", "C4"])]
    print(pd.crosstab(epm.blocker, epm.drug,
                      values=epm.subject_uid, aggfunc=pd.Series.nunique))

    print("\n=== Sex balance (animals) ===")
    print(pd.crosstab(df.cohort, df.sex, values=df.subject_uid,
                      aggfunc=pd.Series.nunique))


def main() -> None:
    DERIVED.mkdir(parents=True, exist_ok=True)
    df = build(DATA_ROOT)
    dest = DERIVED / "manifest.csv"
    df.to_csv(dest, index=False)
    print(f"Wrote {dest} ({len(df)} rows)")

    summarize(df)

    problems = validate(df)
    print("\n=== VALIDATION ===")
    if problems:
        for p in problems:
            print(f"  [!] {p}")
        print("\nKnown, documented gaps (should appear above, harmlessly):")
        for c, m, why in KNOWN_GAPS:
            print(f"  - {c} {m}: {why}")
    else:
        print("  All checks passed.")


main()


Wrote /Users/prestondavis/Desktop/Capstone/derived/manifest.csv (203 rows)

=== Files per cohort x recording type ===
rectype  BL1  BL2  EPM  EZM  inj
cohort                          
C1        14   14    0   14   14
C2        11   11    0   11   11
C3        15   15   15    0   15
C4        11   11   10    0   11

=== ANIMALS per cohort x group (this is your real n) ===
cohort   group  n_mice
    C1     psi       7
    C1     sal       7
    C2     psi       5
    C2     sal       6
    C3     ket       5
    C3     psi       5
    C3     sal       5
    C4 ket_psi       2
    C4 ket_sal       4
    C4 sal_psi       3
    C4 sal_sal       2

=== Pooled 2x2 frame for C3+C4 (animals) ===
drug     psi  sal
blocker          
ket        7    4
sal        8    7

=== Sex balance (animals) ===
sex     F  M
cohort      
C1      3  4
C2      3  3
C3      6  9
C4      7  4

=== VALIDATION ===
  [!] C2: mice with only one drug week: ['F2'] (F2 in C2 is expected and documented)

Known, documented

In [13]:
"""
Stage 2 -- Preprocess and QC every recording.

For each file:
    load -> high-pass -> notch -> anti-aliased downsample -> artifact mask
and write a compressed .npz plus a per-file QC row.

Run:
    python step2_preprocess.py --manifest ../derived/manifest.csv --out ../derived

Design notes
------------
* Artifacts are MASKED, not deleted. Deleting samples breaks time alignment
  with the TTL stream and between regions. Everything downstream must respect
  the mask.
* Thresholds use median/MAD, not mean/SD. Using SD to find outliers is circular:
  the outliers inflate the SD you are comparing them against.
* The QC table is the input to Figure F0. Build F0 before you trust anything.

Dependencies: numpy, scipy, pandas, h5py (+ tqdm optional)
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal

REGIONS = ("BLA", "vHPC", "mPFC")

# numpy 2.0 renamed trapz -> trapezoid; support both.
_trapz = getattr(np, "trapezoid", None) or np.trapz


# ---- Preprocessing parameters. Change here, nowhere else. ----
TARGET_FS = 500.0     # Hz. 500 gives clean access to everything below ~200 Hz.
HIGHPASS_HZ = 0.5     # remove slow drift
LINE_HZ = 60.0        # US mains. VERIFY against a PSD before trusting this.
LINE_HARMONICS = 3    # notch 60, 120, 180
NOTCH_Q = 30.0
ARTIFACT_MAD_Z = 6.0  # robust-z threshold on amplitude and on d/dt
ARTIFACT_PAD_S = 0.25 # dilate each flagged region by this much on each side
FLAT_TOL = 1e-12      # runs with near-zero variance = disconnected channel

PARAMS = dict(
    target_fs=TARGET_FS, highpass_hz=HIGHPASS_HZ, line_hz=LINE_HZ,
    line_harmonics=LINE_HARMONICS, notch_q=NOTCH_Q,
    artifact_mad_z=ARTIFACT_MAD_Z, artifact_pad_s=ARTIFACT_PAD_S,
)


# ----------------------------------------------------------------------
# Loading
# ----------------------------------------------------------------------
def load_recording(path: Path) -> tuple[dict[str, np.ndarray], float]:
    """Return {region: 1-D float array}, fs.

    The Box files are MATLAB v7.3 (.mat stored as HDF5). Also accepts the
    /lfp/<region> HDF5 layout from matlab/export_for_python.m, and older
    .mat files via scipy.io.loadmat.
    """
    path = Path(path)
    import h5py

    def _from_h5(f: "h5py.File"):
        # Exporter layout
        if "fs" in f.attrs:
            fs = float(f.attrs["fs"])
            lfp = {r: np.asarray(f[f"/lfp/{r}"][:], float).ravel()
                   for r in REGIONS if f"/lfp/{r}" in f}
            if lfp:
                return lfp, fs
        # Native Box v7.3: RecData_*/LFP/{BLA,vHPC,mPFC} and RecData_*/meta/Fs
        rec_keys = [k for k in f.keys() if not k.startswith("#")]
        if len(rec_keys) != 1:
            raise ValueError(f"{path.name}: expected 1 RecData variable, got {rec_keys}")
        rec = f[rec_keys[0]]
        fs = float(np.asarray(rec["meta/Fs"]).reshape(-1)[0])
        lfp_g = rec["LFP"]
        lfp = {r: np.asarray(lfp_g[r][:], float).ravel()
               for r in REGIONS if r in lfp_g}
        return lfp, fs

    try:
        with h5py.File(path, "r") as f:
            return _from_h5(f)
    except OSError:
        pass

    from scipy.io import loadmat
    S = loadmat(path, struct_as_record=False, squeeze_me=True)
    key = [k for k in S if not k.startswith("__")]
    if len(key) != 1:
        raise ValueError(f"{path.name}: expected 1 variable, got {key}")
    R = S[key[0]]
    fs = float(np.atleast_1d(R.meta.Fs)[0])
    lfp = {r: np.asarray(getattr(R.LFP, r), float).ravel()
           for r in REGIONS if hasattr(R.LFP, r)}
    return lfp, fs


# ----------------------------------------------------------------------
# Filtering
# ----------------------------------------------------------------------
def preprocess_trace(x: np.ndarray, fs: float) -> tuple[np.ndarray, float]:
    """High-pass, notch, decimate. Returns (clean signal, new fs)."""
    x = np.asarray(x, float)
    x = x - np.nanmedian(x)

    sos = signal.butter(4, HIGHPASS_HZ, btype="highpass", fs=fs, output="sos")
    x = signal.sosfiltfilt(sos, x)          # zero-phase: no phase distortion

    for h in range(1, LINE_HARMONICS + 1):
        f0 = LINE_HZ * h
        if f0 < fs / 2 - 5:
            b, a = signal.iirnotch(f0, NOTCH_Q, fs)
            x = signal.filtfilt(b, a, x)

    q = int(round(fs / TARGET_FS))
    if q > 1:
        x = signal.decimate(x, q, ftype="fir", zero_phase=True)
        fs = fs / q
    return x, fs


# ----------------------------------------------------------------------
# Artifacts
# ----------------------------------------------------------------------
def robust_z(x: np.ndarray) -> np.ndarray:
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    scale = 1.4826 * mad
    if scale <= 0:
        scale = np.finfo(float).eps
    return (x - med) / scale


def artifact_mask(x: np.ndarray, fs: float) -> np.ndarray:
    """True where the sample should be excluded."""
    bad = np.abs(robust_z(x)) > ARTIFACT_MAD_Z

    d = np.diff(x, prepend=x[0])
    bad |= np.abs(robust_z(d)) > ARTIFACT_MAD_Z

    # flat / saturated runs, evaluated in 100 ms blocks
    blk = max(int(0.1 * fs), 1)
    n = (len(x) // blk) * blk
    if n:
        v = x[:n].reshape(-1, blk).var(axis=1)
        flat = np.repeat(v < FLAT_TOL, blk)
        bad[:n] |= flat

    pad = int(ARTIFACT_PAD_S * fs)
    if pad > 0 and bad.any():
        bad = np.convolve(bad.astype(float), np.ones(2 * pad + 1), "same") > 0
    return bad


def band_power(x: np.ndarray, fs: float, good: np.ndarray) -> dict[str, float]:
    """Quick Welch band powers on good samples only -- for QC, not for stats."""
    seg = x[good] if good.sum() > fs * 10 else x
    f, p = signal.welch(seg, fs=fs, nperseg=int(min(len(seg), 4 * fs)))
    out = {}
    for name, (lo, hi) in dict(delta=(1, 4), theta=(4, 12), beta=(13, 30),
                               lgamma=(30, 60)).items():
        m = (f >= lo) & (f < hi)
        out[f"qc_{name}"] = float(_trapz(p[m], f[m])) if m.any() else np.nan
    return out


# ----------------------------------------------------------------------
def process_file(row: pd.Series, outdir: Path) -> dict:
    lfp, fs_in = load_recording(Path(row.path))
    qc: dict = {"stem": row.stem, "fs_in": fs_in}
    clean, masks = {}, {}
    fs_out = fs_in

    for reg in REGIONS:
        if reg not in lfp:
            qc[f"{reg}_present"] = False
            continue
        x, fs_out = preprocess_trace(lfp[reg], fs_in)
        bad = artifact_mask(x, fs_out)
        clean[reg], masks[reg] = x.astype(np.float32), bad

        qc[f"{reg}_present"] = True
        qc[f"{reg}_rms"] = float(np.std(x[~bad])) if (~bad).any() else np.nan
        qc[f"{reg}_pct_artifact"] = float(100 * bad.mean())
        qc[f"{reg}_kurtosis"] = float(pd.Series(x).kurtosis())
        for k, v in band_power(x, fs_out, ~bad).items():
            qc[f"{reg}_{k}"] = v

    qc["fs_out"] = fs_out
    qc["n_samples"] = int(len(next(iter(clean.values())))) if clean else 0
    qc["duration_s"] = qc["n_samples"] / fs_out if fs_out else np.nan

    # union mask -- samples usable for any cross-region measure
    if masks:
        union = np.any(np.vstack(list(masks.values())), axis=0)
        qc["pct_artifact_union"] = float(100 * union.mean())

    np.savez_compressed(
        outdir / f"{row.stem}.npz",
        fs=fs_out,
        **{f"lfp_{r}": clean[r] for r in clean},
        **{f"mask_{r}": masks[r] for r in masks},
    )
    return qc


def main() -> None:
    man = pd.read_csv(DERIVED / "manifest.csv")
    if PROCESS_LIMIT:
        man = man.head(PROCESS_LIMIT)

    cleandir = DERIVED / "clean"
    cleandir.mkdir(parents=True, exist_ok=True)

    rows, failures = [], []
    for i, row in man.iterrows():
        try:
            rows.append(process_file(row, cleandir))
            print(f"[{i+1}/{len(man)}] {row.stem}  ok")
        except Exception as e:                       # noqa: BLE001
            failures.append({"stem": row.stem, "error": repr(e)})
            print(f"[{i+1}/{len(man)}] {row.stem}  FAILED: {e}")

    if not rows:
        raise SystemExit("Every file failed to process -- see the errors above "
                         "and preprocess_failures.csv")

    qc = pd.DataFrame(rows).merge(
        man[["stem", "cohort", "group", "blocker", "drug", "mouse", "sex",
             "rectype", "subject_uid"]],
        on="stem", how="left")
    qc.to_csv(DERIVED / "qc.csv", index=False)
    pd.Series(PARAMS).to_csv(DERIVED / "preprocess_params.csv")
    if failures:
        pd.DataFrame(failures).to_csv(DERIVED / "preprocess_failures.csv", index=False)

    print(f"\nWrote qc.csv ({len(qc)} rows), {len(failures)} failures")
    if "pct_artifact_union" in qc:
        print("\nArtifact % by drug group -- CHECK THIS FOR IMBALANCE:")
        print(qc.groupby(["cohort", "group"])["pct_artifact_union"]
                .describe()[["count", "mean", "max"]].round(2))


main()


[1/203] Psi_C1_psi_F1_BL1  ok
[2/203] Psi_C1_psi_F1_BL2  ok
[3/203] Psi_C1_psi_F1_EZM  ok
[4/203] Psi_C1_psi_F1_inj  ok
[5/203] Psi_C1_psi_F2_BL1  ok
[6/203] Psi_C1_psi_F2_BL2  ok
[7/203] Psi_C1_psi_F2_EZM  ok
[8/203] Psi_C1_psi_F2_inj  ok
[9/203] Psi_C1_psi_F3_BL1  ok
[10/203] Psi_C1_psi_F3_BL2  ok
[11/203] Psi_C1_psi_F3_EZM  ok
[12/203] Psi_C1_psi_F3_inj  ok
[13/203] Psi_C1_psi_M1_BL1  ok
[14/203] Psi_C1_psi_M1_BL2  ok
[15/203] Psi_C1_psi_M1_EZM  ok
[16/203] Psi_C1_psi_M1_inj  ok
[17/203] Psi_C1_psi_M2_BL1  ok
[18/203] Psi_C1_psi_M2_BL2  ok
[19/203] Psi_C1_psi_M2_EZM  ok
[20/203] Psi_C1_psi_M2_inj  ok
[21/203] Psi_C1_psi_M3_BL1  ok
[22/203] Psi_C1_psi_M3_BL2  ok
[23/203] Psi_C1_psi_M3_EZM  ok
[24/203] Psi_C1_psi_M3_inj  ok
[25/203] Psi_C1_psi_M4_BL1  ok
[26/203] Psi_C1_psi_M4_BL2  ok
[27/203] Psi_C1_psi_M4_EZM  ok
[28/203] Psi_C1_psi_M4_inj  ok
[29/203] Psi_C1_sal_F1_BL1  ok
[30/203] Psi_C1_sal_F1_BL2  ok
[31/203] Psi_C1_sal_F1_EZM  ok
[32/203] Psi_C1_sal_F1_inj  ok
[33/203] Psi_C1_s

In [14]:
"""
Stage 3 -- Feature extraction: one row per recording.

Computes, per region: PSD, absolute + relative band power, and (if specparam is
installed) the aperiodic exponent/offset plus aperiodic-adjusted band power.
Per region pair: coherence and wPLI (volume-conduction-robust).
Plus an independent beta-burst re-detection with phase-lag statistics, so you
are not dependent on the upstream analysis you cannot inspect.

Run:
    python step3_features.py --manifest ../derived/manifest.csv --clean ../derived/clean \
                             --out ../derived

Why re-detect bursts when they are already provided?
    Because reproducing a provided result with your own code is the single best
    validation your pipeline can get, and because the provided settings
    (threshold 0.8, min_dur 500 ms) are one arbitrary point in a parameter
    space you should be sweeping.

Dependencies: numpy, scipy, pandas
Optional but recommended: specparam (or fooof)
"""

from __future__ import annotations

import argparse
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import signal

REGIONS = ("BLA", "vHPC", "mPFC")

# numpy 2.0 renamed trapz -> trapezoid; support both.
_trapz = getattr(np, "trapezoid", None) or np.trapz

PAIRS = list(itertools.combinations(REGIONS, 2))

BANDS = {
    "delta": (1, 4),
    "theta": (4, 12),
    "beta": (13, 30),
    "lgamma": (30, 60),
    "hgamma": (60, 100),
}

BURST_BAND = (13, 30)
BURST_PCTILE = 75.0     # amplitude-envelope percentile threshold
BURST_MIN_MS = 150.0    # canonical burst floor; sweep this (see --sweep)
BURST_MERGE_MS = 50.0
APERIODIC_RANGE = (1.0, 45.0)


# ----------------------------------------------------------------------
def load_clean(path: Path):
    z = np.load(path)
    fs = float(z["fs"])
    lfp = {r: z[f"lfp_{r}"] for r in REGIONS if f"lfp_{r}" in z}
    msk = {r: z[f"mask_{r}"] for r in REGIONS if f"mask_{r}" in z}
    return lfp, msk, fs


def psd(x: np.ndarray, fs: float, good: np.ndarray):
    """Welch PSD over good samples. Contiguous good runs are concatenated;
    with 2-s windows the seam artifacts are negligible relative to the gain
    from excluding artifacts."""
    seg = x[good] if good.sum() > 10 * fs else x
    nper = int(min(len(seg), 2 * fs))
    f, p = signal.welch(seg, fs=fs, nperseg=nper, noverlap=nper // 2)
    return f, p


def band_powers(f, p, prefix=""):
    total = _trapz(p[(f >= 1) & (f <= 100)], f[(f >= 1) & (f <= 100)])
    out = {}
    for name, (lo, hi) in BANDS.items():
        m = (f >= lo) & (f < hi)
        val = float(_trapz(p[m], f[m])) if m.any() else np.nan
        out[f"{prefix}abs_{name}"] = val
        out[f"{prefix}rel_{name}"] = val / total if total > 0 else np.nan
    out[f"{prefix}total_1_100"] = float(total)
    return out


def aperiodic(f, p, prefix=""):
    """Fit 1/f with specparam. Returns NaNs (not an error) if unavailable."""
    try:
        try:
            from specparam import SpectralModel as _M
        except ImportError:
            from fooof import FOOOF as _M
        m = _M(peak_width_limits=[2, 12], max_n_peaks=6, verbose=False)
        m.fit(f, p, APERIODIC_RANGE)
        ap = m.aperiodic_params_
        return {
            f"{prefix}ap_offset": float(ap[0]),
            f"{prefix}ap_exponent": float(ap[-1]),
            f"{prefix}ap_r2": float(m.r_squared_),
            f"{prefix}n_peaks": int(m.n_peaks_),
        }
    except Exception:                                  # noqa: BLE001
        return {f"{prefix}ap_offset": np.nan, f"{prefix}ap_exponent": np.nan,
                f"{prefix}ap_r2": np.nan, f"{prefix}n_peaks": np.nan}



def segment_csd(x, y, fs, nperseg, noverlap=None):
    """Per-segment cross-spectral density: returns (freqs, array[n_seg, n_freq])."""
    nperseg = int(nperseg)
    noverlap = nperseg // 2 if noverlap is None else int(noverlap)
    step = nperseg - noverlap
    n = (len(x) - noverlap) // step
    if n < 2:
        f = np.fft.rfftfreq(nperseg, 1 / fs)
        return f, np.zeros((1, len(f)), complex)
    win = np.hanning(nperseg)
    idx = np.arange(nperseg)[None, :] + step * np.arange(n)[:, None]
    X = np.fft.rfft(x[idx] * win, axis=1)
    Y = np.fft.rfft(y[idx] * win, axis=1)
    return np.fft.rfftfreq(nperseg, 1 / fs), X * np.conj(Y)


def connectivity(x: np.ndarray, y: np.ndarray, fs: float, good: np.ndarray, tag: str):
    """Magnitude-squared coherence plus wPLI, band-averaged.

    Report BOTH. Coherence is inflated by volume conduction (a rhythm leaking
    passively from one site to the other looks like perfect zero-lag coupling);
    wPLI discards zero-lag contributions and is the honest number.
    """
    xa = x[good] if good.sum() > 10 * fs else x
    ya = y[good] if good.sum() > 10 * fs else y
    nper = int(min(len(xa), 2 * fs))

    f, cxy = signal.coherence(xa, ya, fs=fs, nperseg=nper, noverlap=nper // 2)

    # wPLI needs the per-segment cross-spectra, which scipy.signal.csd averages
    # away, so we compute them ourselves.
    fz, Pxy = segment_csd(xa, ya, fs, nper)
    im = np.imag(Pxy)
    num = np.abs(np.mean(im, axis=0))
    den = np.mean(np.abs(im), axis=0)
    wpli = np.divide(num, den, out=np.full_like(num, np.nan), where=den > 0)

    out = {}
    for name, (lo, hi) in BANDS.items():
        m = (f >= lo) & (f < hi)
        out[f"{tag}_coh_{name}"] = float(np.nanmean(cxy[m])) if m.any() else np.nan
        mz = (fz >= lo) & (fz < hi)
        out[f"{tag}_wpli_{name}"] = float(np.nanmean(wpli[mz])) if mz.any() else np.nan
    return out


# ----------------------------------------------------------------------
def beta_envelope(x, fs, band=BURST_BAND):
    sos = signal.butter(4, band, btype="bandpass", fs=fs, output="sos")
    xb = signal.sosfiltfilt(sos, x)
    return xb, np.abs(signal.hilbert(xb))


def envelope_threshold(x, fs, good, pctile=BURST_PCTILE, band=BURST_BAND):
    """Amplitude threshold from one recording -- normally that animal's BL1."""
    _, env = beta_envelope(x, fs, band)
    ref = env[good] if good.sum() > 10 * fs else env
    return float(np.percentile(ref, pctile))


def detect_bursts(x, fs, good, band=BURST_BAND, pctile=BURST_PCTILE,
                  min_ms=BURST_MIN_MS, merge_ms=BURST_MERGE_MS, thr=None):
    """Hilbert-envelope burst detection. Returns (starts_s, ends_s, filtered).

    THRESHOLD MODE MATTERS MORE THAN YOU EXPECT.
      thr=None (within-recording percentile): the threshold rescales with the
        recording's own amplitude, so a drug that uniformly raises beta power
        produces the SAME burst rate. You are measuring the temporal structure
        of beta, not how much of it there is. This is a legitimate question,
        but it is not the question most people think they are asking.
      thr=<value from that animal's BL1>: power increases now show up as more
        and longer bursts. This is what you want for a drug-vs-baseline
        comparison. Use --thresh-mode baseline.
    Report which mode you used. Ideally report both.
    """
    xb, env = beta_envelope(x, fs, band)
    if thr is None:
        ref = env[good] if good.sum() > 10 * fs else env
        thr = np.percentile(ref, pctile)

    above = (env > thr) & good
    d = np.diff(above.astype(int), prepend=0, append=0)
    starts = np.flatnonzero(d == 1)
    ends = np.flatnonzero(d == -1)

    # merge gaps shorter than merge_ms
    if len(starts) > 1:
        gap = (starts[1:] - ends[:-1]) / fs * 1000
        keep = np.concatenate([[True], gap > merge_ms])
        new_s, new_e = [], []
        for i, k in enumerate(keep):
            if k:
                new_s.append(starts[i]); new_e.append(ends[i])
            else:
                new_e[-1] = ends[i]
        starts, ends = np.array(new_s), np.array(new_e)

    dur_ms = (ends - starts) / fs * 1000
    ok = dur_ms >= min_ms
    return starts[ok] / fs, ends[ok] / fs, xb


def phase_lag_stats(xb_a, xb_b, fs, starts_s, ends_s):
    """Circular mean phase difference (a minus b) and resultant length per burst.

    Positive mean lag = region A leads region B, matching the upstream
    convention phi_BLA - phi_vHPC.
    """
    pa = np.angle(signal.hilbert(xb_a))
    pb = np.angle(signal.hilbert(xb_b))
    lags, Rs = [], []
    for s, e in zip(starts_s, ends_s):
        i0, i1 = int(s * fs), int(e * fs)
        if i1 - i0 < 2:
            continue
        d = np.angle(np.exp(1j * (pa[i0:i1] - pb[i0:i1])))
        z = np.mean(np.exp(1j * d))
        lags.append(np.angle(z))
        Rs.append(np.abs(z))
    return np.array(lags), np.array(Rs)


def circ_mean(a, w=None):
    if len(a) == 0:
        return np.nan, np.nan
    w = np.ones_like(a) if w is None else np.asarray(w, float)
    z = np.sum(w * np.exp(1j * np.asarray(a))) / np.sum(w)
    return float(np.angle(z)), float(np.abs(z))


# ----------------------------------------------------------------------
def process(stem: str, cleandir: Path, thr: float | None = None,
            session_uid: str | None = None) -> dict:
    lfp, msk, fs = load_clean(cleandir / f"{stem}.npz")
    row: dict = {"stem": stem, "fs": fs,
                 "burst_thresh_mode": "baseline" if thr is not None else "within"}

    goods = {r: ~msk[r] for r in lfp}
    for r, x in lfp.items():
        f, p = psd(x, fs, goods[r])
        row.update(band_powers(f, p, prefix=f"{r}_"))
        row.update(aperiodic(f, p, prefix=f"{r}_"))

    for a, b in PAIRS:
        if a in lfp and b in lfp:
            g = goods[a] & goods[b]
            row.update(connectivity(lfp[a], lfp[b], fs, g, f"{a}-{b}"))

    # Bursts detected on BLA (matching the upstream convention), phase lag
    # measured BLA vs vHPC.
    if "BLA" in lfp:
        s, e, xb_bla = detect_bursts(lfp["BLA"], fs, goods["BLA"], thr=thr)
        dur = (e - s) * 1000
        dur_good = goods["BLA"].sum() / fs
        row["burst_n"] = int(len(s))
        row["burst_rate_per_min"] = 60 * len(s) / dur_good if dur_good else np.nan
        row["burst_dur_mean_ms"] = float(np.mean(dur)) if len(dur) else np.nan
        row["burst_dur_median_ms"] = float(np.median(dur)) if len(dur) else np.nan
        row["burst_occupancy_pct"] = float(100 * dur.sum() / 1000 / dur_good) if dur_good else np.nan

        if "vHPC" in lfp:
            sos = signal.butter(4, BURST_BAND, btype="bandpass", fs=fs, output="sos")
            xb_vh = signal.sosfiltfilt(sos, lfp["vHPC"])
            lags, Rs = phase_lag_stats(xb_bla, xb_vh, fs, s, e)
            mu, R = circ_mean(lags, Rs)
            row["phase_lag_circmean_rad"] = mu
            row["phase_lag_resultant"] = R
            row["phase_R_mean"] = float(np.mean(Rs)) if len(Rs) else np.nan
            row["phase_lag_ms_at_20hz"] = mu / (2 * np.pi) * 50 if np.isfinite(mu) else np.nan
    return row


def main() -> None:
    thresh_mode = "baseline"
    man = pd.read_csv(DERIVED / "manifest.csv")
    if PROCESS_LIMIT:
        man = man.head(PROCESS_LIMIT)
    clean_dir = DERIVED / "clean"

    # Pre-pass: one beta-amplitude threshold per session, taken from its BL1.
    thr_map = {}
    if thresh_mode == "baseline":
        for _, r in man[man.rectype == "BL1"].iterrows():
            try:
                lfp, msk, fs = load_clean(clean_dir / f"{r.stem}.npz")
                if "BLA" in lfp:
                    thr_map[r.session_uid] = envelope_threshold(
                        lfp["BLA"], fs, ~msk["BLA"])
            except Exception:                            # noqa: BLE001
                pass
        print(f"Baseline thresholds computed for {len(thr_map)} sessions")

    rows, fails = [], []
    for i, r in man.iterrows():
        try:
            rows.append(process(r.stem, clean_dir,
                                thr=thr_map.get(r.session_uid),
                                session_uid=r.session_uid))
            print(f"[{i+1}/{len(man)}] {r.stem} ok")
        except Exception as e:                          # noqa: BLE001
            fails.append({"stem": r.stem, "error": repr(e)})
            print(f"[{i+1}/{len(man)}] {r.stem} FAILED: {e}")

    feat = pd.DataFrame(rows).merge(
        man[["stem", "cohort", "group", "blocker", "drug", "mouse", "sex",
             "rectype", "design", "subject_uid", "session_uid"]],
        on="stem", how="left")
    feat.to_csv(DERIVED / "features_session.csv", index=False)
    if fails:
        pd.DataFrame(fails).to_csv(DERIVED / "features_failures.csv", index=False)

    # ---- Normalize every feature to that animal's own BL1 (see guide 5.5) ----
    numeric = [c for c in feat.columns
               if feat[c].dtype.kind == "f" and c not in ("fs",)]
    bl1 = (feat[feat.rectype == "BL1"]
           .set_index("session_uid")[numeric]
           .add_suffix("__bl1"))
    norm = feat.join(bl1, on="session_uid")
    for c in numeric:
        b = norm[f"{c}__bl1"]
        with np.errstate(divide="ignore", invalid="ignore"):
            norm[f"{c}__vs_bl1"] = np.where(
                (b > 0) & (norm[c] > 0), 10 * np.log10(norm[c] / b), norm[c] - b)
    keep = [c for c in norm.columns if not c.endswith("__bl1")]
    norm[keep].to_csv(DERIVED / "features_normalized.csv", index=False)

    print(f"\nWrote features_session.csv ({len(feat)} rows) and features_normalized.csv")
    print(f"{len(fails)} failures")


main()


Baseline thresholds computed for 51 sessions
[1/203] Psi_C1_psi_F1_BL1 ok
[2/203] Psi_C1_psi_F1_BL2 ok
[3/203] Psi_C1_psi_F1_EZM ok
[4/203] Psi_C1_psi_F1_inj ok
[5/203] Psi_C1_psi_F2_BL1 ok
[6/203] Psi_C1_psi_F2_BL2 ok
[7/203] Psi_C1_psi_F2_EZM ok
[8/203] Psi_C1_psi_F2_inj ok
[9/203] Psi_C1_psi_F3_BL1 ok
[10/203] Psi_C1_psi_F3_BL2 ok
[11/203] Psi_C1_psi_F3_EZM ok
[12/203] Psi_C1_psi_F3_inj ok
[13/203] Psi_C1_psi_M1_BL1 ok
[14/203] Psi_C1_psi_M1_BL2 ok
[15/203] Psi_C1_psi_M1_EZM ok
[16/203] Psi_C1_psi_M1_inj ok
[17/203] Psi_C1_psi_M2_BL1 ok
[18/203] Psi_C1_psi_M2_BL2 ok
[19/203] Psi_C1_psi_M2_EZM ok
[20/203] Psi_C1_psi_M2_inj ok
[21/203] Psi_C1_psi_M3_BL1 ok
[22/203] Psi_C1_psi_M3_BL2 ok
[23/203] Psi_C1_psi_M3_EZM ok
[24/203] Psi_C1_psi_M3_inj ok
[25/203] Psi_C1_psi_M4_BL1 ok
[26/203] Psi_C1_psi_M4_BL2 ok
[27/203] Psi_C1_psi_M4_EZM ok
[28/203] Psi_C1_psi_M4_inj ok
[29/203] Psi_C1_sal_F1_BL1 ok
[30/203] Psi_C1_sal_F1_BL2 ok
[31/203] Psi_C1_sal_F1_EZM ok
[32/203] Psi_C1_sal_F1_inj ok
[33/

In [ ]:
"""
Stage 4 -- figures. Dead-channel exclusions ON by default.
Writes to figures_clean/ and leaves figures/ untouched for before/after.
Re-run this cell after changing EXCLUDE in the paths cell.
"""
from step4_figures import run as run_figures

run_figures(
    derived=DERIVED,
    out=FIGURES_CLEAN if EXCLUDE else FIGURES,
    qc_path=DERIVED / "qc.csv",
    exclude=EXCLUDE,
)


In [ ]:
"""
Stage 5 -- stats. Dead-channel exclusions ON by default.
Writes *_clean filenames so previous results/ files are not overwritten.
"""
from step5_stats import run as run_stats

run_stats(
    features=DERIVED / "features_session.csv",
    out=RESULTS,
    metric=STAT_METRIC,
    qc_path=DERIVED / "qc.csv",
    exclude=EXCLUDE,
)
